# Negative-Space Reader: Colab runner

[Open in Colab](https://colab.research.google.com/github/parsa-hke/negative-space-reader/blob/main/notebooks/colab_runner.ipynb)
(always the latest version on `main`; the first time, Colab asks to authorise GitHub: tick *Include private repos*).

Runs one GPU job on Colab against the same data as the Mac (plan §0.6). **Jobs are written on the Mac** in
`configs/colab_jobs.yaml` and pushed to GitHub; this notebook runs the job named under `next:` there, so nothing in
the notebook needs editing: **Runtime > Run all**.

**Once per account:** add these in the key icon on the left (Colab *Secrets*), and give this notebook access:
- `HF_TOKEN`: Hugging Face token with read+write access to the private dataset repo `parsa-hke/negative-space-reader-data`
- `GH_TOKEN`: a fine-grained GitHub token, read-only access to `parsa-hke/negative-space-reader` (contents)
- `OPENROUTER_API_KEY`, `ANTHROPIC_API_KEY`: only for teacher or frontier-model runs

**Every session:** Runtime > Change runtime type > **A100 GPU**, then Runtime > Run all. Outputs go back to the
dataset repo (cell 6); then on the Mac: `.venv/bin/python -m experiments.hub_sync pull --bundles <the job's push>`.
If cell 3 had to upgrade a library and imports fail, use Runtime > Disconnect and delete runtime, then Run all again.

In [ ]:
# 1. GPU and secrets (tokens are put in the environment, never printed)
import os, subprocess
from google.colab import userdata
!nvidia-smi -L
for k in ("HF_TOKEN", "GH_TOKEN", "OPENROUTER_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(f"{k}: not set (only needed for some runs)")
print("secrets loaded:", [k for k in ("HF_TOKEN", "GH_TOKEN", "OPENROUTER_API_KEY", "ANTHROPIC_API_KEY") if os.environ.get(k)])

In [ ]:
# 2. Code: clone (or update) the private repo at the branch to run. The token travels in an HTTP header that git
#    sends to github.com only (set in this machine's ~/.gitconfig, which disappears with the session), never in a URL,
#    so it is not stored in .git/config and cannot appear in an error message; git's own errors are shown, masked.
import base64
BRANCH = "main"
URL = "https://github.com/parsa-hke/negative-space-reader.git"
_auth = base64.b64encode(f"x-access-token:{os.environ['GH_TOKEN'].strip()}".encode()).decode()

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).replace(os.environ["GH_TOKEN"].strip(), "***").replace(_auth, "***")
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}):\n{msg}")
    return r.stdout

git("config", "--global", "http.https://github.com/.extraheader", f"AUTHORIZATION: basic {_auth}")
if not os.path.isdir("/content/negative-space-reader/.git"):
    git("clone", "--quiet", "--branch", BRANCH, URL, "/content/negative-space-reader")
%cd /content/negative-space-reader
git("fetch", "--quiet", "origin"); git("checkout", "--quiet", BRANCH); git("pull", "--quiet", "--ff-only")
print(git("log", "-1", "--oneline"))

In [ ]:
# 3. Libraries: the project's model libraries on top of Colab's own torch, numpy and pandas (kept as Colab ships them).
#    The project package itself is not installed: the code runs from the clone (the working directory).
!pip -q install -r requirements-colab.txt
import sys, numpy, pandas, torch, transformers, sentence_transformers
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__)

In [ ]:
# 4. The job (configs/colab_jobs.yaml, from the clone): JOB = None runs the one under `next:`; or name another job
import yaml
JOB = None
spec = yaml.safe_load(open("configs/colab_jobs.yaml"))
JOB = JOB or spec["next"]
job = spec["jobs"][JOB]
PULL, RUN, PUSH = job["pull"], " ".join(job["run"].split()), job["push"]
print(f"job {JOB}: {job['what']}\n  pull {PULL}\n  run  {RUN}\n  push {PUSH}")
!python -m experiments.hub_sync pull --bundles {PULL}
!python -m pytest -q -x tests/test_skeleton.py

In [ ]:
# 5. The run (experiments/run.py writes runs/<name>_s<seed>/ and a runs/log.csv line)
!python -m experiments.run {RUN}

In [ ]:
# 6. Outputs back to the dataset repo (run folders, pack artefacts, the merged run log); then on the Mac:
#    .venv/bin/python -m experiments.hub_sync pull --bundles <PUSH>
!python -m experiments.hub_sync push --bundles {PUSH}